# Refined P&L Calculation with Automated Tick Data

This notebook automatically fetches calendar spread opportunities and calculates precise P&L using tick data.

**Workflow:**
1. Load pre-downloaded tick data from batch backfill
2. Calculate P&L using realistic execution prices (bid/ask)
3. Compare tick-based vs bar-based P&L
4. Analyze spread costs and slippage

**Prerequisites:**
- Run `dlt-ibapi backfill-batch-calendar-ticks <date> --earnings-timing <timing>` first
- Tick data should be in `data_delta/ticks/`

In [4]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, date, timedelta

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from dlt_ibapi.repositories import (
    OptionTicksReader,
    OptionBarsReader,
)
from dlt_ibapi.strategies import EarningsTimingCalculator

# Rich for better output
from rich import print as rprint
from rich.table import Table
from rich.console import Console

console = Console()

# Set pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

## 1. Configuration

Specify the earnings date and load the IV ranking results CSV.

In [5]:
# Configuration
EARNINGS_DATE = date(2025, 11, 17)
EARNINGS_TIMING = 'PRE_MARKET'

# Load IV ranking results from CSV (generated by notebook 06c)
# If CSV doesn't exist, we'll use manual trade list
results_csv = project_root / "notebooks" / "iv_ratio_ranking_results.csv"

# Data paths
data_dir = project_root / "data_delta"

rprint(f"[bold cyan]Analysis Configuration:[/bold cyan]")
rprint(f"  Earnings Date: {EARNINGS_DATE}")
rprint(f"  Timing: {EARNINGS_TIMING}")
rprint(f"  Data Directory: {data_dir}")
rprint(f"  Results CSV: {results_csv}")

Analysis Configuration:

Earnings Date: 2025-11-17

Timing: PRE_MARKET

Data Directory: /Users/mohamedali/trading_project/dlt-ibapi/data_delta

Results CSV: /Users/mohamedali/trading_project/dlt-ibapi/notebooks/iv_ratio_ranking_results.csv

## 2. Load Opportunities

Load the top calendar spread opportunities from CSV or define manually.

In [6]:
# Use manual trade list (from batch backfill output)
# CSV has different schema - just use hardcoded list for reliability
rprint(f"\n[bold cyan]Using manual trade list for {EARNINGS_DATE} ({EARNINGS_TIMING})[/bold cyan]")

# Manual list of trades (from batch backfill output)
# Format: symbol, strike, short_expiry, long_expiry, right, iv_ratio, bar_pnl
opportunities_df = pd.DataFrame([
    {'symbol': 'ARMK', 'strike': 38.0, 'short_expiry': '2025-11-21', 'long_expiry': '2025-12-19',
     'right': 'C', 'iv_ratio': 2.29, 'bar_pnl': 120.0, 'earnings_timing': 'PRE_MARKET'},
    {'symbol': 'SOHU', 'strike': 15.0, 'short_expiry': '2025-11-21', 'long_expiry': '2025-12-19',
     'right': 'C', 'iv_ratio': 2.26, 'bar_pnl': 0.0, 'earnings_timing': 'PRE_MARKET'},
    {'symbol': 'YSG', 'strike': 7.5, 'short_expiry': '2025-11-21', 'long_expiry': '2026-01-16',
     'right': 'C', 'iv_ratio': 2.01, 'bar_pnl': -30.0, 'earnings_timing': 'PRE_MARKET'},
    {'symbol': 'JJSF', 'strike': 85.0, 'short_expiry': '2025-11-21', 'long_expiry': '2025-12-19',
     'right': 'C', 'iv_ratio': 2.00, 'bar_pnl': 120.0, 'earnings_timing': 'PRE_MARKET'},
    {'symbol': 'NIU', 'strike': 5.0, 'short_expiry': '2025-11-21', 'long_expiry': '2025-12-19',
     'right': 'C', 'iv_ratio': 1.78, 'bar_pnl': 5.0, 'earnings_timing': 'PRE_MARKET'},
])

# Convert dates
opportunities_df['short_expiry'] = pd.to_datetime(opportunities_df['short_expiry']).dt.date
opportunities_df['long_expiry'] = pd.to_datetime(opportunities_df['long_expiry']).dt.date
opportunities_df['earnings_date'] = EARNINGS_DATE

# Display opportunities
if len(opportunities_df) > 0:
    rprint("\n[bold cyan]Opportunities to analyze:[/bold cyan]")
    display_cols = ['symbol', 'strike', 'right', 'iv_ratio', 'bar_pnl', 'earnings_timing']
    rprint(opportunities_df[display_cols].to_string(index=False))
else:
    rprint("[bold red]No opportunities found![/bold red]")

Using manual trade list for 2025-11-17 (PRE_MARKET)

Opportunities to analyze:

symbol  strike right  iv_ratio  bar_pnl earnings_timing
  ARMK    38.0     C      2.29    120.0      PRE_MARKET
  SOHU    15.0     C      2.26      0.0      PRE_MARKET
   YSG     7.5     C      2.01    -30.0      PRE_MARKET
  JJSF    85.0     C      2.00    120.0      PRE_MARKET
   NIU     5.0     C      1.78      5.0      PRE_MARKET

## 3. Helper Functions

In [7]:
def calculate_execution_price(ticks_df, action='buy'):
    """Calculate execution price from tick data."""
    if len(ticks_df) == 0:
        return {
            'first_tick': None,
            'time_weighted_avg': None,
            'worst_case': None,
            'best_case': None,
            'median': None,
            'avg_spread': None,
            'avg_spread_pct': None,
            'num_ticks': 0
        }
    
    # Select price based on action
    if action == 'buy':
        price_col = 'ask_price'
        worst_fn = max
        best_fn = min
    else:  # sell
        price_col = 'bid_price'
        worst_fn = min
        best_fn = max
    
    prices = ticks_df[price_col].dropna()
    
    if len(prices) == 0:
        return {
            'first_tick': None,
            'time_weighted_avg': None,
            'worst_case': None,
            'best_case': None,
            'median': None,
            'avg_spread': None,
            'avg_spread_pct': None,
            'num_ticks': len(ticks_df)
        }
    
    return {
        'first_tick': prices.iloc[0],
        'time_weighted_avg': prices.mean(),
        'worst_case': worst_fn(prices),
        'best_case': best_fn(prices),
        'median': prices.median(),
        'avg_spread': ticks_df['spread'].mean(),
        'avg_spread_pct': ticks_df['spread_pct'].mean() if 'spread_pct' in ticks_df.columns else None,
        'num_ticks': len(ticks_df)
    }

## 4. Analyze Each Opportunity

Load tick data and calculate P&L for each calendar spread.

In [8]:
# Initialize readers
tick_reader = OptionTicksReader(str(data_dir), "ticks")
calculator = EarningsTimingCalculator()

# Storage for results
all_results = []

for idx, opp in opportunities_df.iterrows():
    symbol = opp['symbol']
    strike = opp['strike']
    short_expiry = opp['short_expiry'] if isinstance(opp['short_expiry'], date) else pd.to_datetime(opp['short_expiry']).date()
    long_expiry = opp['long_expiry'] if isinstance(opp['long_expiry'], date) else pd.to_datetime(opp['long_expiry']).date()
    right = opp['right']
    earnings_timing = opp['earnings_timing']
    
    # Calculate windows
    windows = calculator.calculate_windows(
        earnings_date=EARNINGS_DATE,
        earnings_time=earnings_timing,
    )
    
    rprint(f"\n[bold cyan]{'='*80}[/bold cyan]")
    rprint(f"[bold cyan]Analyzing: {symbol} ${strike}{right}[/bold cyan]")
    rprint(f"  Short: {short_expiry}, Long: {long_expiry}")
    rprint(f"  Entry: {windows.entry.start} to {windows.entry.end}")
    rprint(f"  Exit: {windows.exit.start} to {windows.exit.end}")
    rprint(f"  IV Ratio: {opp['iv_ratio']:.3f}")
    rprint(f"  Bar P&L: ${opp['bar_pnl']:.2f}")
    
    try:
        # Load tick data for both legs
        short_entry = tick_reader.get_ticks(symbol, short_expiry, strike, right, windows.entry.start, windows.entry.end, 'bid_ask')
        short_exit = tick_reader.get_ticks(symbol, short_expiry, strike, right, windows.exit.start, windows.exit.end, 'bid_ask')
        long_entry = tick_reader.get_ticks(symbol, long_expiry, strike, right, windows.entry.start, windows.entry.end, 'bid_ask')
        long_exit = tick_reader.get_ticks(symbol, long_expiry, strike, right, windows.exit.start, windows.exit.end, 'bid_ask')
        
        rprint(f"\n[bold green]Ticks loaded:[/bold green] Short({len(short_entry)}, {len(short_exit)}), Long({len(long_entry)}, {len(long_exit)})")
        
        # Calculate execution prices
        se_prices = calculate_execution_price(short_entry, 'sell')  # Sell short
        sx_prices = calculate_execution_price(short_exit, 'buy')    # Buy to close
        le_prices = calculate_execution_price(long_entry, 'buy')    # Buy long
        lx_prices = calculate_execution_price(long_exit, 'sell')    # Sell to close
        
        # Calculate calendar spread P&L
        if (se_prices['time_weighted_avg'] and sx_prices['time_weighted_avg'] and
            le_prices['time_weighted_avg'] and lx_prices['time_weighted_avg']):
            
            # Entry: Long - Short (net debit)
            entry_net = le_prices['time_weighted_avg'] - se_prices['time_weighted_avg']
            # Exit: Long - Short (net credit)
            exit_net = lx_prices['time_weighted_avg'] - sx_prices['time_weighted_avg']
            
            # P&L
            pnl_per_contract = exit_net - entry_net
            pnl_total = pnl_per_contract * 100
            
            # Spread costs
            entry_spread = (se_prices['avg_spread'] or 0) + (le_prices['avg_spread'] or 0)
            exit_spread = (sx_prices['avg_spread'] or 0) + (lx_prices['avg_spread'] or 0)
            total_spread_cost = (entry_spread + exit_spread) * 100
            
            net_pnl = pnl_total - total_spread_cost
            
            rprint(f"\n[bold cyan]Results:[/bold cyan]")
            rprint(f"  Entry Net: ${entry_net:.4f}")
            rprint(f"  Exit Net: ${exit_net:.4f}")
            rprint(f"  Tick P&L: ${pnl_total:.2f}")
            rprint(f"  Spread Cost: ${total_spread_cost:.2f}")
            rprint(f"  Net P&L: ${net_pnl:.2f}")
            rprint(f"  vs Bar P&L: ${pnl_total - opp['bar_pnl']:.2f} difference")
            
            all_results.append({
                'symbol': symbol,
                'strike': strike,
                'right': right,
                'iv_ratio': opp['iv_ratio'],
                'bar_pnl': opp['bar_pnl'],
                'tick_pnl': pnl_total,
                'spread_cost': total_spread_cost,
                'net_pnl': net_pnl,
                'difference': pnl_total - opp['bar_pnl'],
                'total_ticks': len(short_entry) + len(short_exit) + len(long_entry) + len(long_exit),
            })
        else:
            rprint("[bold red]Insufficient tick data[/bold red]")
            
    except Exception as e:
        rprint(f"[bold red]Error: {str(e)}[/bold red]")

================================================================================

Analyzing: ARMK $38.0C

Short: 2025-11-21, Long: 2025-12-19

Entry: 2025-11-14 15:00:00 to 2025-11-14 16:00:00

Exit: 2025-11-17 09:00:00 to 2025-11-17 10:00:00

...

Exit: 2025-11-17 09:00:00 to 2025-11-17 10:00:00

IV Ratio: 1.780

Bar P&L: $5.00

Ticks loaded: Short(0, 0), Long(0, 0)

Insufficient tick data

## 5. Summary Table

In [9]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    rprint("\n[bold cyan]" + "="*100 + "[/bold cyan]")
    rprint("[bold cyan]SUMMARY: TICK-BASED P&L[/bold cyan]")
    rprint("[bold cyan]" + "="*100 + "[/bold cyan]\n")
    
    table = Table(title="Calendar Spread P&L Analysis", show_header=True)
    table.add_column("Symbol", style="cyan bold")
    table.add_column("Strike", justify="right")
    table.add_column("IV Ratio", justify="right", style="green")
    table.add_column("Bar P&L", justify="right", style="yellow")
    table.add_column("Tick P&L", justify="right", style="cyan")
    table.add_column("Spread Cost", justify="right", style="red")
    table.add_column("Net P&L", justify="right", style="bold green")
    table.add_column("Diff", justify="right")
    table.add_column("Ticks", justify="right", style="dim")
    
    for _, row in results_df.iterrows():
        diff_pct = (row['difference'] / row['bar_pnl'] * 100) if row['bar_pnl'] != 0 else 0
        table.add_row(
            row['symbol'],
            f"${row['strike']:.1f}",
            f"{row['iv_ratio']:.3f}",
            f"${row['bar_pnl']:.2f}",
            f"${row['tick_pnl']:.2f}",
            f"${row['spread_cost']:.2f}",
            f"${row['net_pnl']:.2f}",
            f"${row['difference']:.2f}",
            f"{int(row['total_ticks'])}"
        )
    
    console.print(table)
    
    rprint(f"\n[bold cyan]Statistics:[/bold cyan]")
    rprint(f"  Mean Tick P&L: ${results_df['tick_pnl'].mean():.2f}")
    rprint(f"  Mean Spread Cost: ${results_df['spread_cost'].mean():.2f}")
    rprint(f"  Mean Net P&L: ${results_df['net_pnl'].mean():.2f}")
    rprint(f"  Best: {results_df.loc[results_df['net_pnl'].idxmax()]['symbol']} (${results_df['net_pnl'].max():.2f})")
    
else:
    rprint("\n[bold red]No results![/bold red]")
    rprint("[bold yellow]Make sure to run:[/bold yellow]")
    rprint(f"  dlt-ibapi backfill-batch-calendar-ticks {EARNINGS_DATE.strftime('%Y-%m-%d')} --earnings-timing {EARNINGS_TIMING}")

No results!

Make sure to run:

dlt-ibapi backfill-batch-calendar-ticks 2025-11-17 --earnings-timing PRE_MARKET

## 6. Key Insights

In [10]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    spread_pct = (results_df['spread_cost'].mean() / results_df['tick_pnl'].abs().mean()) * 100
    
    rprint("\n[bold cyan]KEY INSIGHTS:[/bold cyan]")
    rprint(f"  • Spread costs average {spread_pct:.1f}% of gross P&L")
    rprint(f"  • Tick-based P&L differs from bar-based by ${results_df['difference'].mean():.2f} on average")
    rprint(f"  • Average {int(results_df['total_ticks'].mean())} ticks per opportunity")
    rprint("\n[bold green]✓ Tick data provides more realistic P&L estimates[/bold green]")
    rprint("[bold green]✓ Spread costs are material and should be factored in[/bold green]")